In [1]:
%env WORKDIR=/tmp/vault
%env CERT_NAME=2026

env: WORKDIR=/tmp/vault
env: CERT_NAME=2026


In [2]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

## https://developer.hashicorp.com/vault/tutorials/secrets-management/pki-engine

![image.png](attachment:image.png)

In [3]:
! curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request POST --data '{"hmac":false}' $VAULT_ADDR/v1/sys/config/auditing/request-headers/my-header

## Step 1: Generate root CA

### Enable PKI engine==mount point

In [4]:
! vault secrets enable pki 

Success! Enabled the pki secrets engine at: pki/


In [5]:
# Tune PKI to set max_tll
! vault secrets tune -max-lease-ttl=87600h pki

Success! Tuned the secrets engine at: pki/


### Generate rootCA

In [6]:
%%bash

vault write -field=certificate pki/root/generate/internal \
     common_name="example.com" alt_names="test.com" \
     issuer_name="root-$CERT_NAME" \
     ttl=87600h > ${WORKDIR}/root_$CERT_NAME_ca.crt

### # List CA information

In [7]:
%%bash
# List CA information
export ISSUER=$(vault list -format=json pki/issuers/ | jq -r .[0])
echo $ISSUER

vault read pki/issuer/$ISSUER | tail -n 11

09ef9884-3c97-2106-4cec-2d2288b1834b
enable_time_checks                   false
issuer_id                            09ef9884-3c97-2106-4cec-2d2288b1834b
issuer_name                          root-2026
issuing_certificates                 []
key_id                               8b112d10-2d6f-98d0-8143-3c9d0120f96e
leaf_not_after_behavior              err
manual_chain                         <nil>
ocsp_servers                         []
revocation_signature_algorithm       SHA256WithRSA
revoked                              false
usage                                crl-signing,issuing-certificates,ocsp-signing,read-only


[PKI Role](https://developer.hashicorp.com/vault/api-docs/secret/pki#create-update-role) details

In [8]:
%%bash
# Create a role that will allow for using certificates (in this case any name will be valid)
vault write pki/roles/$CERT_NAME-servers allow_any_name=true no_store=false

Key                                   Value
---                                   -----
allow_any_name                        true
allow_bare_domains                    false
allow_glob_domains                    false
allow_ip_sans                         true
allow_localhost                       true
allow_subdomains                      false
allow_token_displayname               false
allow_wildcard_certificates           true
allowed_domains                       []
allowed_domains_template              false
allowed_other_sans                    []
allowed_serial_numbers                []
allowed_uri_sans                      []
allowed_uri_sans_template             false
allowed_user_ids                      []
basic_constraints_valid_for_non_ca    false
client_flag                           true
cn_validations                        [email hostname]
code_signing_flag                     false
country                               []
email_protection_flag                 false


### Configure CA and CRL URLs

In [9]:
%%bash
# Configure Vault cluster URLs
vault write pki/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki

Key         Value
---         -----
aia_path    https://vault.vault.svc.cluster.local:8200/v1/pki
path        https://vault.vault.svc.cluster.local:8200/v1/pki


In [10]:
%%bash

vault write pki/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

Key                              Value
---                              -----
crl_distribution_points          [{{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der]
delta_crl_distribution_points    []
enable_templating                true
issuing_certificates             [{{cluster_aia_path}}/issuer/{{issuer_id}}/der]
ocsp_servers                     [{{cluster_path}}/ocsp]


[OCSP](https://developer.hashicorp.com/vault/api-docs/secret/pki#ocsp-request)

## Step 2: Generate intermediate CA

### The intermediate CA is expressed a another PKI engine with a separate mount point

In [11]:
! vault secrets enable -path=pki_int pki

Success! Enabled the pki secrets engine at: pki_int/


In [12]:
# the mount is configured with a max_tll
! vault secrets tune -max-lease-ttl=43800h pki_int

Success! Tuned the secrets engine at: pki_int/


### Generate Intermediate CA whose CSR is going to be signed by the root CA at pki mount path

In [13]:
%%bash

vault write -format=json pki_int/intermediate/generate/internal \
     common_name="example.com Intermediate Authority" \
     issuer_name="example-dot-com-intermediate" \
     | jq -r '.data.csr' > $WORKDIR/pki_intermediate.csr

### Send intermediateCA CSR for signing with CA mount point

In [14]:
%%bash

vault write -format=json pki/root/sign-intermediate \
     issuer_ref="root-$CERT_NAME" \
     csr=@$WORKDIR/pki_intermediate.csr \
     format=pem_bundle ttl="43800h" \
     | jq -r '.data.certificate' > ${WORKDIR}/intermediate.cert.pem

### Import signed intermediate CA (`intermediate.cer.pem`) to its correspondant mount point

In [15]:
# Import signed intermediate CA to its correspondant mount point
! vault write pki_int/intermediate/set-signed certificate=@${WORKDIR}/intermediate.cert.pem

WARNING! The following warnings were returned from Vault:

  * This mount hasn't configured any authority information access (AIA)
  fields; this may make it harder for systems to find missing certificates
  in the chain or to validate revocation status of certificates. Consider
  updating /config/urls or the newly generated issuer with this information.

Key                 Value
---                 -----
existing_issuers    <nil>
existing_keys       <nil>
imported_issuers    [ecdaeb82-0870-319d-7d3e-d227574032a8 ee3e0f81-96b6-d39e-8d2d-7fccb055a674]
imported_keys       <nil>
mapping             map[ecdaeb82-0870-319d-7d3e-d227574032a8:0e49808f-26d1-2a35-790c-f32453fb863c ee3e0f81-96b6-d39e-8d2d-7fccb055a674:]


### Configure CA and CRL URLs for intermediate CA

In [16]:
%%bash
# Configure Vault cluster URLs
vault write pki_int/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki_int \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki_int

Key         Value
---         -----
aia_path    https://vault.vault.svc.cluster.local:8200/v1/pki_int
path        https://vault.vault.svc.cluster.local:8200/v1/pki_int


In [17]:
%%bash
vault write pki_int/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

Key                              Value
---                              -----
crl_distribution_points          [{{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der]
delta_crl_distribution_points    []
enable_templating                true
issuing_certificates             [{{cluster_aia_path}}/issuer/{{issuer_id}}/der]
ocsp_servers                     [{{cluster_path}}/ocsp]


## Step 3: Create role -> https://developer.hashicorp.com/vault/tutorials/secrets-management/pki-engine#step-3-create-a-role

### Create role that allow certificates to be signed for domain `example.com` and `test.com` with intermediate CA (`pki_int` mount point)

In [18]:
%%bash
# Create role that allow certificates to be signed for domain `example.com` and `test.com`
vault write pki_int/roles/example-dot-com \
     issuer_ref="$(vault read -field=default pki_int/config/issuers)" \
     allowed_domains="example.com","test.com" \
     allow_subdomains=true \
     allow_glob_domains=true \
     allow_wildcard_certificates=true \
     allow_ip_sans=true \
     allowed_uri_sans="*.example.com" \
     max_ttl="24h" \
     ttl="12h" \
     ext_key_usage="Client Auth"

Key                                   Value
---                                   -----
allow_any_name                        false
allow_bare_domains                    false
allow_glob_domains                    true
allow_ip_sans                         true
allow_localhost                       true
allow_subdomains                      true
allow_token_displayname               false
allow_wildcard_certificates           true
allowed_domains                       [example.com test.com]
allowed_domains_template              false
allowed_other_sans                    []
allowed_serial_numbers                []
allowed_uri_sans                      [*.example.com]
allowed_uri_sans_template             false
allowed_user_ids                      []
basic_constraints_valid_for_non_ca    false
client_flag                           true
cn_validations                        [email hostname]
code_signing_flag                     false
country                               []
email_protec

## Step 4: Requests certificates

### Generate Certificates using Vault CLI

In [19]:
! vault write pki_int/issue/example-dot-com common_name="*.test.com" ip_sans="8.8.8.9" \
uri_sans="otrauri.example.com,masuri.example.com" ttl="1m" 

Key                 Value
---                 -----
authority_key_id    ac:d3:fd:d2:f4:57:ac:ec:9e:3c:11:b5:d4:94:3b:6c:d3:4a:98:84
ca_chain            [-----BEGIN CERTIFICATE-----
MIIEdjCCA16gAwIBAgIUduOdhzSiNBGVHFmZG7cVb1KHHc4wDQYJKoZIhvcNAQEL
BQAwFjEUMBIGA1UEAxMLZXhhbXBsZS5jb20wHhcNMjYwOTE4MTM0NTE2WhcNMzEw
OTE3MTM0NTQ2WjAtMSswKQYDVQQDEyJleGFtcGxlLmNvbSBJbnRlcm1lZGlhdGUg
QXV0aG9yaXR5MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAy7iYTrLd
KXHj+y+16ZEaNWeMB9Rdsxf4FFw9F4wrq1/aCV+hcxYH+GploSTG8byG21gEAs6P
3gGhE0hGYXoRADOSlBFdtxQHO3VN0epG5qTaujJxuZCLVcGNhLO9DPEhCS/FqLCr
o6UlLpy/KBCfkeVkKtm6jXeokuB1/yoX2nAnyekR/AIxeB2lm+pfL7otP6avBJIQ
66h8acXoIyUy04CcCy20lSI/Fd48pGPmAr7MNtGMI3j2E2w/nhf2a2VUKZqRZzGe
9vloqw2MQQ8vRi8Qhbb3WyMB6EnF9MvfPi6I16p3NIEvM6nnHhXg2f6W6/CwLGX3
MXHZdZ95t0btIwIDAQABo4IBozCCAZ8wDgYDVR0PAQH/BAQDAgEGMA8GA1UdEwEB
/wQFMAMBAf8wHQYDVR0OBBYEFKzT/dL0V6zsnjwRtdSUO2zTSpiEMB8GA1UdIwQY
MBaAFGPQdSKOi/1kQZCLBWzDoRbpIbkIMIHDBggrBgEFBQcBAQSBtjCBszBCBggr
BgEFBQcwAYY2aHR0cHM6Ly92YXVsdC52YXVsdC5

### if you want to see details on the audit logs about the certificate information unhash

In [20]:
%%bash

# Tune to unhash request and response values
vault secrets tune  \
     -max-lease-ttl="43800h"  -audit-non-hmac-request-keys="csr" -audit-non-hmac-request-keys="certificate" -audit-non-hmac-request-keys=issuer_ref -audit-non-hmac-request-keys="common_name" -audit-non-hmac-request-keys=alt_names -audit-non-hmac-request-keys=other_sans  \
     -audit-non-hmac-request-keys="ip_sans" -audit-non-hmac-request-keys=uri_sans  -audit-non-hmac-request-keys=ttl  -audit-non-hmac-request-keys=not_after  \
     -audit-non-hmac-request-keys=serial_number -audit-non-hmac-request-keys=key_type -audit-non-hmac-request-keys=private_key_format \
     -audit-non-hmac-request-keys=ou -audit-non-hmac-request-keys=organization -audit-non-hmac-request-keys=country \
     -audit-non-hmac-request-keys=locality -audit-non-hmac-request-keys=province -audit-non-hmac-request-keys=street_address \
     -audit-non-hmac-request-keys=postal_code -audit-non-hmac-request-keys=permitted_dns_domains -audit-non-hmac-request-keys=policy_identitiers \
     -audit-non-hmac-request-keys=ext_key_usage_oids -audit-non-hmac-response-keys=certificate -audit-non-hmac-response-keys=issuing_ca -audit-non-hmac-response-keys=error  \
     -audit-non-hmac-response-keys=serial_number -audit-non-hmac-response-keys=ca_chain -audit-non-hmac-response-keys=private_key_type -audit-non-hmac-response-keys=expiration pki_int

Success! Tuned the secrets engine at: pki_int/


### Generate Certificates using the API

In [21]:
%%bash

curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request PUT --silent --data '{"common_name": "hash123.example.com", "ttl": "2h"}' $VAULT_ADDR/v1/pki_int/issue/example-dot-com | jq

{
  "request_id": "c8273757-b99d-438d-9b6a-b1a522522488",
  "lease_id": "",
  "renewable": false,
  "lease_duration": 0,
  "data": {
    "authority_key_id": "ac:d3:fd:d2:f4:57:ac:ec:9e:3c:11:b5:d4:94:3b:6c:d3:4a:98:84",
    "ca_chain": [
      "-----BEGIN CERTIFICATE-----\nMIIEdjCCA16gAwIBAgIUduOdhzSiNBGVHFmZG7cVb1KHHc4wDQYJKoZIhvcNAQEL\nBQAwFjEUMBIGA1UEAxMLZXhhbXBsZS5jb20wHhcNMjYwOTE4MTM0NTE2WhcNMzEw\nOTE3MTM0NTQ2WjAtMSswKQYDVQQDEyJleGFtcGxlLmNvbSBJbnRlcm1lZGlhdGUg\nQXV0aG9yaXR5MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAy7iYTrLd\nKXHj+y+16ZEaNWeMB9Rdsxf4FFw9F4wrq1/aCV+hcxYH+GploSTG8byG21gEAs6P\n3gGhE0hGYXoRADOSlBFdtxQHO3VN0epG5qTaujJxuZCLVcGNhLO9DPEhCS/FqLCr\no6UlLpy/KBCfkeVkKtm6jXeokuB1/yoX2nAnyekR/AIxeB2lm+pfL7otP6avBJIQ\n66h8acXoIyUy04CcCy20lSI/Fd48pGPmAr7MNtGMI3j2E2w/nhf2a2VUKZqRZzGe\n9vloqw2MQQ8vRi8Qhbb3WyMB6EnF9MvfPi6I16p3NIEvM6nnHhXg2f6W6/CwLGX3\nMXHZdZ95t0btIwIDAQABo4IBozCCAZ8wDgYDVR0PAQH/BAQDAgEGMA8GA1UdEwEB\n/wQFMAMBAf8wHQYDVR0OBBYEFKzT/dL0V6zsnjwRtdSUO2zTSpiEMB8GA1UdIwQY\n

In [22]:
%%bash
export CERT_NAME=san.example.com

# Using CURL
curl -k --header "X-Vault-Token: $VAULT_TOKEN"\
    --request POST --silent \
    --data '{"common_name": "'"$CERT_NAME"'", "ttl": "1m"}' \
    $VAULT_ADDR/v1/pki_int/issue/example-dot-com > ${WORKDIR}/mycert.json

cat ${WORKDIR}/mycert.json | jq -r .data.certificate | openssl x509 -in /dev/stdin -text -noout

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            38:28:ac:18:c8:6d:76:49:d3:fd:40:2e:0f:5c:3b:64:7d:97:34:d8
    Signature Algorithm: sha256WithRSAEncryption
        Issuer: CN=example.com Intermediate Authority
        Validity
            Not Before: Sep 18 13:45:17 2026 GMT
            Not After : Sep 18 13:46:47 2026 GMT
        Subject: CN=san.example.com
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:ad:4c:d6:d0:81:54:75:d4:b3:8f:64:09:98:18:
                    a0:13:d7:a0:6f:24:c8:40:73:86:f7:97:d9:00:1a:
                    c7:f8:ff:88:41:2c:e9:de:53:01:97:15:3a:a6:b4:
                    0d:d6:58:c2:5b:6e:95:48:b0:5f:88:3d:e3:74:3b:
                    03:53:5d:db:24:78:2f:8f:04:80:b9:bd:68:8e:ff:
                    29:db:44:52:b6:aa:ff:23:8b:62:bd:b5:7f:47:af:
                    ba:39:57:4b:3a:e4:66:51:fc:3c:94:3a:1a

In [23]:
%%bash
export CERT_NAME="test107.test.com"

#Using CURL
curl -k --header "X-Vault-Token: $VAULT_TOKEN"\
    --request POST --silent\
    --data '{"common_name": "'"$CERT_NAME"'", "ttl": "1m"}' \
    $VAULT_ADDR/v1/pki_int/issue/example-dot-com > ${WORKDIR}/mycert.json

cat ${WORKDIR}/mycert.json | jq -r .data.certificate > ${WORKDIR}/mycert_leaf.pem
cat ${WORKDIR}/mycert.json | jq -r .data.private_key > ${WORKDIR}/mycert_key.pem
openssl x509 -in  ${WORKDIR}/mycert_leaf.pem -text -noout

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            21:f3:d0:b0:ca:8a:d0:a3:29:40:78:ab:74:96:af:a3:ec:89:9d:fe
    Signature Algorithm: sha256WithRSAEncryption
        Issuer: CN=example.com Intermediate Authority
        Validity
            Not Before: Sep 18 13:45:17 2026 GMT
            Not After : Sep 18 13:46:47 2026 GMT
        Subject: CN=test107.test.com
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:d0:a4:1e:11:6c:88:b3:f2:86:63:6e:bb:3e:59:
                    da:73:3a:3d:72:ec:56:3e:15:8a:f5:13:7b:b1:91:
                    ff:56:26:9e:fb:6b:71:18:0f:48:da:71:11:55:cd:
                    0a:41:e3:1b:39:63:ae:d9:34:c3:6a:bc:d6:70:b4:
                    a5:4d:74:28:25:f1:05:8c:5b:1f:88:f1:17:1a:0b:
                    aa:3a:41:be:ef:87:81:8c:50:05:94:3d:73:e8:35:
                    68:a6:a8:63:d6:33:18:17:05:8f:08:22:0

## Read Certificates

### List Certificates

In [24]:
%%bash


curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys

[
  "21:f3:d0:b0:ca:8a:d0:a3:29:40:78:ab:74:96:af:a3:ec:89:9d:fe",
  "38:28:ac:18:c8:6d:76:49:d3:fd:40:2e:0f:5c:3b:64:7d:97:34:d8",
  "4c:a9:68:de:29:c1:f0:34:40:a6:14:5c:ee:e6:4f:9b:3d:87:cc:1e",
  "53:a6:08:71:1b:6d:a1:65:33:c1:bd:8c:5a:50:01:e1:aa:b3:aa:f5"
]


### Read Details about a certificate based on serial number
> #### Note private key can just be retrieved at creation time

In [25]:
%%bash
export SERIAL=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN"\
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[0])
    

curl -k --silent\
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request GET \
    $VAULT_ADDR/v1/pki_int/cert/$SERIAL | jq -r .data.certificate > ${WORKDIR}/temp.pem
    
openssl x509 -in  ${WORKDIR}/temp.pem -text -noout 

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            21:f3:d0:b0:ca:8a:d0:a3:29:40:78:ab:74:96:af:a3:ec:89:9d:fe
    Signature Algorithm: sha256WithRSAEncryption
        Issuer: CN=example.com Intermediate Authority
        Validity
            Not Before: Sep 18 13:45:17 2026 GMT
            Not After : Sep 18 13:46:47 2026 GMT
        Subject: CN=test107.test.com
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:d0:a4:1e:11:6c:88:b3:f2:86:63:6e:bb:3e:59:
                    da:73:3a:3d:72:ec:56:3e:15:8a:f5:13:7b:b1:91:
                    ff:56:26:9e:fb:6b:71:18:0f:48:da:71:11:55:cd:
                    0a:41:e3:1b:39:63:ae:d9:34:c3:6a:bc:d6:70:b4:
                    a5:4d:74:28:25:f1:05:8c:5b:1f:88:f1:17:1a:0b:
                    aa:3a:41:be:ef:87:81:8c:50:05:94:3d:73:e8:35:
                    68:a6:a8:63:d6:33:18:17:05:8f:08:22:0

### Revoke certificate based on serial number

In [26]:
%%bash

# https://developer.hashicorp.com/vault/api-docs/secret/pki#revoke-certificate
export SERIAL=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN" \
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[3])
    
curl -k --silent --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data '{"serial_number": "'"$SERIAL"'"}' \
    $VAULT_ADDR/v1/pki_int/revoke | jq

{
  "request_id": "189260ec-a366-b264-c0a3-58796235b686",
  "lease_id": "",
  "renewable": false,
  "lease_duration": 0,
  "data": {
    "revocation_time": 1789739147,
    "revocation_time_rfc3339": "2026-09-18T13:45:47.9412288Z",
    "state": "revoked"
  },
  "wrap_info": null,
  "warnings": null,
  "auth": null,
  "mount_type": "pki"
}


In [27]:
%%bash
export SERIAL2=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN" \
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[1])

vault write pki_int/revoke serial_number=$SERIAL2

Key                        Value
---                        -----
revocation_time            1789739148
revocation_time_rfc3339    2026-09-18T13:45:48.067771806Z
state                      revoked


### Let's check the CRL

In [28]:
%%bash
curl -k --silent  $VAULT_ADDR/v1/pki_int/crl -o $WORKDIR/out.crl
openssl crl -inform DER -text -noout -in $WORKDIR/out.crl

Certificate Revocation List (CRL):
        Version 2 (0x1)
    Signature Algorithm: sha256WithRSAEncryption
        Issuer: /CN=example.com Intermediate Authority
        Last Update: Sep 18 13:45:48 2026 GMT
        Next Update: Sep 21 13:45:48 2026 GMT
        CRL extensions:
            X509v3 Authority Key Identifier: 
                keyid:AC:D3:FD:D2:F4:57:AC:EC:9E:3C:11:B5:D4:94:3B:6C:D3:4A:98:84

            X509v3 CRL Number: 
                5
Revoked Certificates:
    Serial Number: 3828AC18C86D7649D3FD402E0F5C3B647D9734D8
        Revocation Date: Sep 18 13:45:48 2026 GMT
    Serial Number: 53A608711B6DA16533C1BD8C5A5001E1AAB3AAF5
        Revocation Date: Sep 18 13:45:47 2026 GMT
    Signature Algorithm: sha256WithRSAEncryption
         14:84:e3:8b:31:cc:87:4b:3c:2c:f5:30:c5:2a:dc:06:ba:bf:
         cf:83:e2:97:e0:ad:97:fe:5f:73:db:c9:8c:bd:76:73:ba:34:
         11:eb:c3:14:92:18:e9:43:a7:b6:f9:27:10:5c:00:c9:79:33:
         d8:00:c6:0b:b2:3c:63:4e:dc:69:0b:e6:bf:cf:73:63:9a

### Vault does not remove the certificate from its list of `cert store` until a tidy operation is run

In [29]:
%%bash
echo "Todos los certificados"
curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys

Todos los certificados
[
  "21:f3:d0:b0:ca:8a:d0:a3:29:40:78:ab:74:96:af:a3:ec:89:9d:fe",
  "38:28:ac:18:c8:6d:76:49:d3:fd:40:2e:0f:5c:3b:64:7d:97:34:d8",
  "4c:a9:68:de:29:c1:f0:34:40:a6:14:5c:ee:e6:4f:9b:3d:87:cc:1e",
  "53:a6:08:71:1b:6d:a1:65:33:c1:bd:8c:5a:50:01:e1:aa:b3:aa:f5"
]


### Two types of `tidy` operation one-of or automatic

In [30]:
%%bash
# https://developer.hashicorp.com/vault/api-docs/secret/pki#tidy
vault write pki_int/tidy tidy_cert_store=true tidy_revoked_certs=true safety_buffer=1m

# Auto Tidy
vault write pki_int/config/auto-tidy tidy_cert_store=true tidy_revoked_certs=true safety_buffer=60m

WARNING! The following warnings were returned from Vault:

  * Tidy operation successfully started. Any information from the operation
  will be printed to Vault's server logs.



Key                                         Value
---                                         -----
acme_account_safety_buffer                  2592000
enabled                                     false
interval_duration                           12h
issuer_safety_buffer                        31536000
maintain_stored_certificate_counts          false
max_startup_backoff_duration                15m
min_startup_backoff_duration                5m
pause_duration                              0s
publish_stored_certificate_count_metrics    false
revocation_queue_safety_buffer              172800
safety_buffer                               3600
tidy_acme                                   false
tidy_cert_metadata                          false
tidy_cert_store                             true
tidy_cmpv2_nonce_store                      false
tidy_cross_cluster_revoked_certs            false
tidy_expired_issuers                        false
tidy_move_legacy_ca_bundle                  false
tidy_r

### List Certificates

In [31]:
%%bash

echo "Todos los certificados"
curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys



Todos los certificados
[
  "21:f3:d0:b0:ca:8a:d0:a3:29:40:78:ab:74:96:af:a3:ec:89:9d:fe",
  "38:28:ac:18:c8:6d:76:49:d3:fd:40:2e:0f:5c:3b:64:7d:97:34:d8",
  "4c:a9:68:de:29:c1:f0:34:40:a6:14:5c:ee:e6:4f:9b:3d:87:cc:1e",
  "53:a6:08:71:1b:6d:a1:65:33:c1:bd:8c:5a:50:01:e1:aa:b3:aa:f5"
]


## Clean UP

In [32]:
! vault audit enable file file_path=stdout

Success! Enabled the file audit device at: file/


# EST Testing

[Enrollment over Secure Transport (EST)](https://developer.hashicorp.com/vault/docs/secrets/pki/est) (RFC 7030) is a Vault **Enterprise** PKI protocol that lets devices obtain CA certificates and enroll / re-enroll client certificates.

This section stands up a **dedicated intermediate mount** (`pki_est`) and wires:

1. **HTTP Basic** authentication (`userpass`, batch tokens)
2. **TLS client-certificate** authentication (`cert`, batch tokens)
3. **`simpleenroll`** and **`simplereenroll`** on `/.well-known/est/`

EST clients call `https://<vault>/.well-known/est/{cacerts,simpleenroll,simplereenroll}` (not `/v1/...`). Vault authenticates each request by delegating to those auth mounts, then returns a base64-encoded PKCS#7 (CMS certs-only) payload.

> Requires Vault Enterprise. Only one PKI mount in the cluster can set `default_mount=true`.

## Step 1: Dedicated intermediate CA (`pki_est`)

Reuse the existing root at `pki/` to sign a new intermediate used **only** for EST issuance. Keep it separate from `pki_int` so EST policy, roles, and auth accessors do not collide with the earlier demo.

### Enable the EST PKI mount and raise `max_lease_ttl`

In [33]:
%%bash
mkdir -p ${WORKDIR}/est

# Vault EST returns MIME base64 PKCS#7 with CRLF (RFC 7030).
# macOS LibreSSL `openssl base64 -d -A` yields an empty decode on that format;
# wrapping as PEM PKCS#7 is portable.
cat > ${WORKDIR}/est/p7_to_pem.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
{
  printf '%s\n' '-----BEGIN PKCS7-----'
  tr -d '\r' < "$1"
  printf '%s\n' '-----END PKCS7-----'
} | openssl pkcs7 -inform PEM -print_certs -out "$2"
EOF
chmod +x ${WORKDIR}/est/p7_to_pem.sh

vault secrets enable -path=pki_est pki 2>/dev/null || echo "pki_est/ already enabled"
vault secrets tune -max-lease-ttl=43800h pki_est

Success! Enabled the pki secrets engine at: pki_est/
Success! Tuned the secrets engine at: pki_est/


### Generate the intermediate CSR (key stays inside Vault)

In [34]:
%%bash
vault write -format=json pki_est/intermediate/generate/internal \
     common_name="example.com EST Intermediate Authority" \
     issuer_name="est-intermediate" \
     | jq -r '.data.csr' > ${WORKDIR}/est/pki_est_intermediate.csr

openssl req -in ${WORKDIR}/est/pki_est_intermediate.csr -noout -subject

subject=/CN=example.com EST Intermediate Authority


### Sign the EST intermediate with the root CA at `pki/`

In [35]:
%%bash
vault write -format=json pki/root/sign-intermediate \
     issuer_ref="root-$CERT_NAME" \
     csr=@${WORKDIR}/est/pki_est_intermediate.csr \
     format=pem_bundle ttl="43800h" \
     | jq -r '.data.certificate' > ${WORKDIR}/est/pki_est_intermediate.cert.pem

openssl x509 -in ${WORKDIR}/est/pki_est_intermediate.cert.pem -noout -subject -issuer

subject= /CN=example.com EST Intermediate Authority
issuer= /CN=example.com


### Import the signed intermediate into `pki_est`

In [36]:
# Import signed EST intermediate CA
! vault write pki_est/intermediate/set-signed certificate=@${WORKDIR}/est/pki_est_intermediate.cert.pem

WARNING! The following warnings were returned from Vault:

  * This mount hasn't configured any authority information access (AIA)
  fields; this may make it harder for systems to find missing certificates
  in the chain or to validate revocation status of certificates. Consider
  updating /config/urls or the newly generated issuer with this information.

Key                 Value
---                 -----
existing_issuers    <nil>
existing_keys       <nil>
imported_issuers    [712c876b-964a-7f1a-067f-a88527642022 d0aff9d5-60bb-34cf-bc4d-7afa4a20292c]
imported_keys       <nil>
mapping             map[712c876b-964a-7f1a-067f-a88527642022:8928edce-808d-1044-50a7-f32c0971b79c d0aff9d5-60bb-34cf-bc4d-7afa4a20292c:]


### Configure cluster and AIA URLs for the EST intermediate

In [37]:
%%bash
vault write pki_est/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki_est \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki_est

vault write pki_est/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

vault read pki_est/config/cluster
vault read -format=json pki_est/cert/ca | jq -r .data.certificate > ${WORKDIR}/est/pki_est_ca.pem
vault read -format=json pki/cert/ca | jq -r .data.certificate > ${WORKDIR}/est/root_ca.pem
cat ${WORKDIR}/est/pki_est_ca.pem ${WORKDIR}/est/root_ca.pem > ${WORKDIR}/est/ca_chain.pem
echo "EST intermediate CA subject:"
openssl x509 -in ${WORKDIR}/est/pki_est_ca.pem -noout -subject

Key         Value
---         -----
aia_path    https://vault.vault.svc.cluster.local:8200/v1/pki_est
path        https://vault.vault.svc.cluster.local:8200/v1/pki_est
Key                              Value
---                              -----
crl_distribution_points          [{{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der]
delta_crl_distribution_points    []
enable_templating                true
issuing_certificates             [{{cluster_aia_path}}/issuer/{{issuer_id}}/der]
ocsp_servers                     [{{cluster_path}}/ocsp]
Key         Value
---         -----
aia_path    https://vault.vault.svc.cluster.local:8200/v1/pki_est
path        https://vault.vault.svc.cluster.local:8200/v1/pki_est
EST intermediate CA subject:
subject= /CN=example.com EST Intermediate Authority


## Step 2: EST issuance role

Role-based path policy (`role:est-clients`) restricts CNs/SANs that EST will sign. `use_csr_common_name` / `use_csr_sans` must be true so the PKCS#10 request drives identity. `Client Auth` EKU is required so issued certs can authenticate via the `cert` method and, on `/simplereenroll`, be presented in the TLS handshake ([RFC 7030 §3.3.2](https://www.rfc-editor.org/rfc/rfc7030#section-3.3.2)).

In [38]:
%%bash
vault write pki_est/roles/est-clients \
     issuer_ref="$(vault read -field=default pki_est/config/issuers)" \
     allowed_domains="example.com,est.example.com" \
     allow_subdomains=true \
     allow_bare_domains=true \
     allow_glob_domains=true \
     allow_ip_sans=true \
     client_flag=true \
     server_flag=false \
     key_type=rsa \
     key_bits=2048 \
     max_ttl="24h" \
     ttl="12h" \
     not_before_duration="0s" \
     require_cn=true \
     use_csr_common_name=true \
     use_csr_sans=true \
     no_store=false \
     ext_key_usage="Client Auth"

vault read pki_est/roles/est-clients

Key                                   Value
---                                   -----
allow_any_name                        false
allow_bare_domains                    true
allow_glob_domains                    true
allow_ip_sans                         true
allow_localhost                       true
allow_subdomains                      true
allow_token_displayname               false
allow_wildcard_certificates           true
allowed_domains                       [example.com est.example.com]
allowed_domains_template              false
allowed_other_sans                    []
allowed_serial_numbers                []
allowed_uri_sans                      []
allowed_uri_sans_template             false
allowed_user_ids                      []
basic_constraints_valid_for_non_ca    false
client_flag                           true
cn_validations                        [email hostname]
code_signing_flag                     false
country                               []
email_protection_fl

## Step 3: EST authentication (HTTP Basic + TLS)

Vault EST does **not** consume `X-Vault-Token`. It delegates to dedicated auth mounts in the same namespace:

| EST credential | Vault auth mount | Token type |
| --- | --- | --- |
| HTTP Basic (`Authorization: Basic ...`) | `est-userpass` (`userpass`) | **batch** |
| TLS client certificate | `est-cert` (`cert`) | **batch** |

Batch tokens are mandatory: every EST request is authenticated, and service tokens would leak leases. ACL paths must match the **internal redirected** PKI path (`pki_est/roles/est-clients/est/...`), not `/.well-known/est/`.

### Policy for role-based EST enroll / re-enroll

In [39]:
%%bash
vault policy write est-enroll - <<'EOF'
path "pki_est/roles/est-clients/est/simpleenroll" {
  capabilities = ["create", "update"]
}
path "pki_est/roles/est-clients/est/simplereenroll" {
  capabilities = ["create", "update"]
}
EOF

vault policy read est-enroll

Success! Uploaded policy: est-enroll
path "pki_est/roles/est-clients/est/simpleenroll" {
  capabilities = ["create", "update"]
}
path "pki_est/roles/est-clients/est/simplereenroll" {
  capabilities = ["create", "update"]
}


### HTTP Basic: dedicated `userpass` mount (`est-userpass`)

In [40]:
%%bash
vault auth enable -path=est-userpass userpass 2>/dev/null || echo "est-userpass/ already enabled"
vault auth tune -token-type=batch est-userpass

vault write auth/est-userpass/users/estuser \
     password="estpass" \
     token_policies="est-enroll" \
     token_type="batch" \
     token_ttl="5m"

echo "userpass accessor: $(vault read -field=accessor sys/auth/est-userpass)"
vault read sys/auth/est-userpass

Success! Enabled userpass auth method at: est-userpass/
Success! Tuned the auth method at: est-userpass/
Success! Data written to: auth/est-userpass/users/estuser
userpass accessor: auth_userpass_8132326c
Key                        Value
---                        -----
accessor                   auth_userpass_8132326c
config                     map[default_lease_ttl:0 force_no_cache:false max_lease_ttl:0 token_type:batch]
deprecation_status         supported
description                n/a
external_entropy_access    false
local                      false
options                    <nil>
plugin_version             n/a
running_plugin_version     v2.1.1+builtin.vault
running_sha256             n/a
seal_wrap                  false
type                       userpass
uuid                       0f21fa77-f8c5-1506-d330-6cccdf480097


### TLS: dedicated `cert` mount (`est-cert`)

Trust the EST intermediate CA so any leaf it issues (matching `*.est.example.com`) can authenticate. `cert_role=est-clients` is later passed from the EST config as the `name` parameter on cert login.

In [41]:
%%bash
vault auth enable -path=est-cert cert 2>/dev/null || echo "est-cert/ already enabled"
vault auth tune -token-type=batch est-cert

vault write auth/est-cert/certs/est-clients \
     display_name="est-tls" \
     policies="est-enroll" \
     certificate=@${WORKDIR}/est/pki_est_ca.pem \
     allowed_common_names="*.est.example.com" \
     token_policies="est-enroll" \
     token_type="batch" \
     token_ttl="5m"

echo "cert accessor: $(vault read -field=accessor sys/auth/est-cert)"
vault read auth/est-cert/certs/est-clients

Success! Enabled cert auth method at: est-cert/
Success! Tuned the auth method at: est-cert/
Success! Data written to: auth/est-cert/certs/est-clients
cert accessor: auth_cert_b2769637
Key                             Value
---                             -----
alias_metadata                  map[]
allowed_common_names            [*.est.example.com]
allowed_dns_sans                <nil>
allowed_email_sans              <nil>
allowed_metadata_extensions     <nil>
allowed_names                   <nil>
allowed_organizational_units    <nil>
allowed_organizations           <nil>
allowed_uri_sans                <nil>
certificate                     -----BEGIN CERTIFICATE-----
MIIEejCCA2KgAwIBAgIUGR/3Rm++7Dybnsuq7KAx5U4j+ogwDQYJKoZIhvcNAQEL
BQAwFjEUMBIGA1UEAxMLZXhhbXBsZS5jb20wHhcNMjYwOTE4MTM0NTE5WhcNMzEw
OTE3MTM0NTQ5WjAxMS8wLQYDVQQDEyZleGFtcGxlLmNvbSBFU1QgSW50ZXJtZWRp
YXRlIEF1dGhvcml0eTCCASIwDQYJKoZIhvcNAQEBBQADggEPADCCAQoCggEBANcA
tvJgZez0WSvGWIglLqK07rnZOkEdpUFPWhlV2QXiUyq0Z9L0VbiqejHKvYb4jUL

## Step 4: Tune the PKI mount and enable EST

1. Allow EST response headers (`Content-Transfer-Encoding`, `Content-Length`, `WWW-Authenticate`)
2. Register both auth accessors as `delegated-auth-accessors`
3. Enable EST, register the default `/.well-known/est/` prefix, and bind authenticators

See the [EST configuration API](https://developer.hashicorp.com/vault/api-docs/secret/pki/issuance#set-est-configuration).

In [42]:
%%bash
USERPASS_ACCESSOR=$(vault read -field=accessor sys/auth/est-userpass)
CERT_ACCESSOR=$(vault read -field=accessor sys/auth/est-cert)

echo "USERPASS_ACCESSOR=${USERPASS_ACCESSOR}"
echo "CERT_ACCESSOR=${CERT_ACCESSOR}"

vault secrets tune \
  -allowed-response-headers="Content-Transfer-Encoding" \
  -allowed-response-headers="Content-Length" \
  -allowed-response-headers="WWW-Authenticate" \
  -delegated-auth-accessors="${USERPASS_ACCESSOR}" \
  -delegated-auth-accessors="${CERT_ACCESSOR}" \
  pki_est

vault write pki_est/config/est - <<EOF
{
  "enabled": true,
  "default_mount": true,
  "default_path_policy": "role:est-clients",
  "label_to_path_policy": {
    "est-clients": "role:est-clients"
  },
  "authenticators": {
    "cert": {
      "accessor": "${CERT_ACCESSOR}",
      "cert_role": "est-clients"
    },
    "userpass": {
      "accessor": "${USERPASS_ACCESSOR}"
    }
  },
  "enable_sentinel_parsing": true,
  "audit_fields": ["common_name", "alt_names", "ip_sans", "uri_sans", "csr"]
}
EOF

USERPASS_ACCESSOR=auth_userpass_8132326c
CERT_ACCESSOR=auth_cert_b2769637
Success! Tuned the secrets engine at: pki_est/
Key                        Value
---                        -----
audit_fields               [common_name alt_names ip_sans uri_sans csr]
authenticators             map[cert:map[accessor:auth_cert_b2769637 cert_role:est-clients] userpass:map[accessor:auth_userpass_8132326c]]
default_mount              true
default_path_policy        role:est-clients
enable_sentinel_parsing    true
enabled                    true
label_to_path_policy       map[est-clients:role:est-clients]
last_updated               2026-09-18T13:45:51Z


### Verify EST configuration (enabled, default mount, both authenticators)

In [43]:
%%bash
vault read -format=json pki_est/config/est | jq .

echo
echo "=== delegated auth accessors + allowed response headers ==="
vault read -format=json sys/mounts/pki_est | jq '{
  delegated_auth_accessors: .data.config.delegated_auth_accessors,
  allowed_response_headers: .data.config.allowed_response_headers
}'

{
  "request_id": "4755e106-dc8d-5daf-86b3-c1e6dfb2cfe6",
  "lease_id": "",
  "lease_duration": 0,
  "renewable": false,
  "data": {
    "audit_fields": [
      "common_name",
      "alt_names",
      "ip_sans",
      "uri_sans",
      "csr"
    ],
    "authenticators": {
      "cert": {
        "accessor": "auth_cert_b2769637",
        "cert_role": "est-clients"
      },
      "userpass": {
        "accessor": "auth_userpass_8132326c"
      }
    },
    "default_mount": true,
    "default_path_policy": "role:est-clients",
    "enable_sentinel_parsing": true,
    "enabled": true,
    "label_to_path_policy": {
      "est-clients": "role:est-clients"
    },
    "last_updated": "2026-09-18T13:45:51Z"
  },
  "warnings": null,
  "mount_type": "pki"
}

=== delegated auth accessors + allowed response headers ===
{
  "delegated_auth_accessors": [
    "auth_userpass_8132326c",
    "auth_cert_b2769637"
  ],
  "allowed_response_headers": [
    "Content-Transfer-Encoding",
    "Content-Length",
  

## Step 5: Verify `cacerts` (unauthenticated)

`GET /cacerts` must work **without** credentials and return the CA chain as PKCS#7. Exercise both the RFC path and the role-qualified API path.

Vault sends MIME base64 with CRLF (76-column). On macOS LibreSSL, `openssl base64 -d -A` produces an empty decode — wrap the body as PEM PKCS#7 (`p7_to_pem.sh`) instead.

In [44]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

cat > ${WORKDIR}/est/p7_to_pem.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
{
  printf '%s\n' '-----BEGIN PKCS7-----'
  tr -d '\r' < "$1"
  printf '%s\n' '-----END PKCS7-----'
} | openssl pkcs7 -inform PEM -print_certs -out "$2"
EOF
chmod +x ${WORKDIR}/est/p7_to_pem.sh

echo "=== GET /.well-known/est/cacerts ==="
curl -k -sS -D ${WORKDIR}/est/cacerts.hdr -o ${WORKDIR}/est/cacerts.p7 \
  "${EST_BASE}/cacerts"
echo "--- response headers ---"
grep -iE 'HTTP/|content-type|content-transfer-encoding|content-length' ${WORKDIR}/est/cacerts.hdr

echo
echo "=== CA certificates from PKCS#7 ==="
# MIME base64 + CRLF: do not use `openssl base64 -d -A` on macOS LibreSSL
${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/cacerts.p7 ${WORKDIR}/est/cacerts.pem
grep -E 'subject=|issuer=' ${WORKDIR}/est/cacerts.pem || true
echo
openssl x509 -in ${WORKDIR}/est/cacerts.pem -noout -subject -issuer -dates

echo
echo "=== GET /v1/pki_est/roles/est-clients/est/cacerts ==="
curl -k -sS -D ${WORKDIR}/est/cacerts_api.hdr -o ${WORKDIR}/est/cacerts_api.p7 \
  "${VAULT_ADDR}/v1/pki_est/roles/est-clients/est/cacerts"
grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/cacerts_api.hdr
echo "well-known bytes=$(wc -c < ${WORKDIR}/est/cacerts.p7) api bytes=$(wc -c < ${WORKDIR}/est/cacerts_api.p7)" 

=== GET /.well-known/est/cacerts ===
--- response headers ---
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime
content-length: 2788

=== CA certificates from PKCS#7 ===
subject=/CN=example.com EST Intermediate Authority
issuer=/CN=example.com
subject=/CN=example.com
issuer=/CN=example.com

subject= /CN=example.com EST Intermediate Authority
issuer= /CN=example.com
notBefore=Sep 18 13:45:19 2026 GMT
notAfter=Sep 17 13:45:49 2031 GMT

=== GET /v1/pki_est/roles/est-clients/est/cacerts ===
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime
well-known bytes=    2788 api bytes=    2788


## Step 6: Enrollment and re-enrollment with HTTP Basic

RFC 7030: POST a base64 DER PKCS#10 (`Content-Type: application/pkcs10`) to `simpleenroll` / `simplereenroll`.

[Section 2](https://www.rfc-editor.org/rfc/rfc7030#section-2) is an informative operational overview. The protocol requirements are in [§3](https://www.rfc-editor.org/rfc/rfc7030#section-3) and [§4](https://www.rfc-editor.org/rfc/rfc7030#section-4). [§4.2.2](https://www.rfc-editor.org/rfc/rfc7030#section-4.2.2) requires the CSR Subject and SubjectAltName to be identical to the certificate being renewed. Vault checks that constraint when the current leaf is in the TLS handshake.

When both credentials are present, HTTP Basic selects the Vault auth mount that issues the token. `/simplereenroll` still requires that leaf.

Credentials: `estuser` / `estpass`.

### Basic auth — `simpleenroll`

In [45]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="device1.est.example.com"

openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/device1.key \
  -out ${WORKDIR}/est/device1.csr \
  -subj "/CN=${CN}" 2>/dev/null
openssl req -in ${WORKDIR}/est/device1.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/device1.p10

echo "=== CSR ==="
openssl req -in ${WORKDIR}/est/device1.csr -noout -subject

echo
echo "=== POST /.well-known/est/simpleenroll (HTTP Basic) ==="
curl -k -sS -D ${WORKDIR}/est/enroll_basic.hdr \
  -o ${WORKDIR}/est/device1.p7 \
  --user "estuser:estpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"

echo "--- response headers ---"
grep -iE 'HTTP/|content-type|content-transfer-encoding|www-authenticate' ${WORKDIR}/est/enroll_basic.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/enroll_basic.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST enroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/device1.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/device1.p7 ${WORKDIR}/est/device1.pem
# First PEM block is the issued leaf
openssl x509 -in ${WORKDIR}/est/device1.pem -out ${WORKDIR}/est/device1_leaf.pem
cp ${WORKDIR}/est/device1.key ${WORKDIR}/est/device1_leaf.key

echo
echo "=== issued certificate ==="
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -noout -subject -issuer -serial -dates
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -text -noout | grep -E "Public Key Algorithm:|Signature Algorithm:|X509v3 Extended Key Usage:|TLS Web Client" -A1

echo
echo "=== openssl verify against EST CA chain ==="
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/device1_leaf.pem

echo
echo "serial stored for re-enroll comparison:"
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -noout -serial | tee ${WORKDIR}/est/device1.serial

=== CSR ===
subject=/CN=device1.est.example.com

=== POST /.well-known/est/simpleenroll (HTTP Basic) ===
--- response headers ---
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime; smime-type=certs-only

=== issued certificate ===
subject= /CN=device1.est.example.com
issuer= /CN=example.com EST Intermediate Authority
serial=72EA52E2D4DC6AC2DE72FC2E8A3EA19BC1B54F5A
notBefore=Sep 18 13:45:21 2026 GMT
notAfter=Sep 19 01:45:51 2026 GMT
    Signature Algorithm: sha256WithRSAEncryption
        Issuer: CN=example.com EST Intermediate Authority
--
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
--
            X509v3 Extended Key Usage: 
                TLS Web Client Authentication
            X509v3 Subject Key Identifier: 
--
    Signature Algorithm: sha256WithRSAEncryption
         62:b7:3a:0b:bb:4e:46:2c:b6:70:5c:46:cf:33:64:f4:de:39:

=== openssl verify against EST CA chain ===
/tmp/vault/est/device1_leaf.pem: OK


### Basic-only `simplereenroll` — expected failure (no TLS client cert)

[Vault EST docs](https://developer.hashicorp.com/vault/docs/secrets/pki/est) say HTTP Basic is **preferred over TLS client certs for authentication**. That selects the delegated `userpass` mount (`simpleenroll` with Basic returns 200).

`/simplereenroll` still requires the **current leaf in the TLS handshake**. [RFC 7030 §4.2.2](https://www.rfc-editor.org/rfc/rfc7030#section-4.2.2) requires the CSR Subject and SubjectAltName to be identical to the certificate being renewed. Without that certificate, Vault returns **`400 no certificate found in TLS state`**. The handshake also proves possession of the current private key. The same requirement is what [Keyfactor](https://docs.keyfactor.com/ejbca/latest/est-client-mode-configuration) and [DigiCert](https://docs.digicert.com/en/device-trust-manager/tutorials/configure-and-use-est.html) document for `simplereenroll`.

This role issues `Client Auth`, so the certificate can be used for TLS client authentication and [§3.3.2](https://www.rfc-editor.org/rfc/rfc7030#section-3.3.2) requires the client to present it. [§2.3](https://www.rfc-editor.org/rfc/rfc7030#section-2.3) is informative overview text for a certificate that cannot be used for TLS client authentication. Its example is another certificate, and §4.2.2 still applies to the CSR. A Basic-only call is outside that text.

The next cell is the working path: Basic (ACL) **plus** the existing client cert (the certificate being renewed).

In [46]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== POST /.well-known/est/simplereenroll (HTTP Basic ONLY — no client cert) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_basic_only.hdr \
  -o ${WORKDIR}/est/reenroll_basic_only.body \
  --user "estuser:estpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type' ${WORKDIR}/est/reenroll_basic_only.hdr
echo "body: $(cat ${WORKDIR}/est/reenroll_basic_only.body)"

HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_basic_only.hdr)
if [ "${HTTP_CODE}" = "400" ] && grep -q "no certificate found in TLS state" ${WORKDIR}/est/reenroll_basic_only.body; then
  echo "EXPECTED: Basic authenticates, but simplereenroll still requires a TLS client certificate"
else
  echo "UNEXPECTED: wanted HTTP 400 + 'no certificate found in TLS state', got HTTP ${HTTP_CODE}"
  exit 1
fi

=== POST /.well-known/est/simplereenroll (HTTP Basic ONLY — no client cert) ===
HTTP/2 400 
content-type: text/plain; charset=utf-8
body: no certificate found in TLS state
EXPECTED: Basic authenticates, but simplereenroll still requires a TLS client certificate


### Basic auth — `simplereenroll` (same identity, new key, new serial)

HTTP Basic is the Vault authenticator (ACL). `--cert/--key` present the certificate being renewed. [RFC 7030 §4.2.2](https://www.rfc-editor.org/rfc/rfc7030#section-4.2.2) requires the CSR Subject and SubjectAltName to be identical to that certificate; the handshake proves possession of its private key. Without the leaf: `no certificate found in TLS state`.

Vault enforces that match on the encoded fields:

| Check | Failure message | Fix in the CSR |
| --- | --- | --- |
| Subject | `CSR Subject field does not match client certificate` | `string_mask = nombstr` so `CN` is `PrintableString` (LibreSSL `-subj` emits `UTF8String`) |
| SAN extension | `CSR SubjectAltName Extension does not match client certificate` | `subjectAltName = DNS:<CN>` **and** `openssl req -reqexts ext` (LibreSSL ignores `req_extensions` in the config otherwise) |

The CSR uses a **new key** (rekey). The cell verifies the renewed cert is bound to it, then rotates `device1_leaf.pem` / `device1.key` so the cell can be re-run.

In [47]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="device1.est.example.com"
OLD_SERIAL=$(cat ${WORKDIR}/est/device1.serial)

# Rekey CSR: Subject and SAN must be identical to the current leaf (RFC 7030 §4.2.2).
# Vault compares the encoded fields (PrintableString CN, DNS SAN).
#  - string_mask=nombstr -> CN as PrintableString (LibreSSL `-subj` emits UTF8String)
#  - -reqexts ext        -> LibreSSL omits req_extensions from the config unless passed on the CLI
#  - subjectAltName=DNS  -> Vault compares the SAN extension on reenroll
cat > ${WORKDIR}/est/reenroll_req.cnf <<EOF
[req]
distinguished_name = dn
req_extensions = ext
string_mask = nombstr
prompt = no
utf8 = no
[dn]
CN = ${CN}
[ext]
subjectAltName = DNS:${CN}
EOF
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/device1_re.key \
  -out ${WORKDIR}/est/device1_re.csr \
  -config ${WORKDIR}/est/reenroll_req.cnf \
  -reqexts ext
echo "=== renewal CSR (expect PRINTABLESTRING + DNS SAN) ==="
openssl asn1parse -in ${WORKDIR}/est/device1_re.csr | grep PRINTABLESTRING || true
openssl req -in ${WORKDIR}/est/device1_re.csr -noout -text | grep -A1 "Subject Alternative Name"
openssl req -in ${WORKDIR}/est/device1_re.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/device1_re.p10

echo "=== POST /.well-known/est/simplereenroll (HTTP Basic + current client cert) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_basic.hdr \
  -o ${WORKDIR}/est/device1_re.p7 \
  --user "estuser:estpass" \
  --cert ${WORKDIR}/est/device1_leaf.pem \
  --key ${WORKDIR}/est/device1.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1_re.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/reenroll_basic.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_basic.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST reenroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/device1_re.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/device1_re.p7 ${WORKDIR}/est/device1_re.pem
openssl x509 -in ${WORKDIR}/est/device1_re.pem -out ${WORKDIR}/est/device1_re_leaf.pem

NEW_SERIAL=$(openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -serial)
echo "old ${OLD_SERIAL}"
echo "new ${NEW_SERIAL}"
openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -subject -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/device1_re_leaf.pem

if [ -z "${NEW_SERIAL}" ] || [ "${OLD_SERIAL}" = "${NEW_SERIAL}" ]; then
  echo "FAIL: re-enroll did not issue a new serial"
  exit 1
fi

CERT_PUB=$(openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -pubkey | openssl md5)
KEY_PUB=$(openssl rsa -in ${WORKDIR}/est/device1_re.key -pubout 2>/dev/null | openssl md5)
if [ "${CERT_PUB}" != "${KEY_PUB}" ]; then
  echo "FAIL: renewed certificate does not match device1_re.key"
  exit 1
fi
echo "OK: re-enroll issued a new serial bound to the new key"

cp ${WORKDIR}/est/device1_re_leaf.pem ${WORKDIR}/est/device1_leaf.pem
cp ${WORKDIR}/est/device1_re.key ${WORKDIR}/est/device1.key
echo "${NEW_SERIAL}" > ${WORKDIR}/est/device1.serial

Generating a 2048 bit RSA private key
..............................................................+++++
...............................................................+++++
writing new private key to '/tmp/vault/est/device1_re.key'
-----


=== renewal CSR (expect PRINTABLESTRING + DNS SAN) ===
   22:d=5  hl=2 l=  23 prim: PRINTABLESTRING   :device1.est.example.com
            X509v3 Subject Alternative Name: 
                DNS:device1.est.example.com
=== POST /.well-known/est/simplereenroll (HTTP Basic + current client cert) ===
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime; smime-type=certs-only
old serial=72EA52E2D4DC6AC2DE72FC2E8A3EA19BC1B54F5A
new serial=09901BF02FC97B9063398F9E0C52372AAD671843
subject= /CN=device1.est.example.com
notBefore=Sep 18 13:45:22 2026 GMT
notAfter=Sep 19 01:45:52 2026 GMT
/tmp/vault/est/device1_re_leaf.pem: OK
OK: re-enroll issued a new serial bound to the new key


## Step 7: Enrollment and re-enrollment with TLS client certificates

Bootstrap a factory credential with the admin token (`pki_est/issue/est-clients`). That cert authenticates subsequent EST calls via `curl --cert/--key` (no Basic header — otherwise userpass would win).

### Bootstrap a TLS client cert, then EST `simpleenroll`

In [48]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== bootstrap leaf via PKI issue (admin token) ==="
vault write -format=json pki_est/issue/est-clients \
     common_name="bootstrap.est.example.com" \
     ttl="12h" \
     | tee ${WORKDIR}/est/bootstrap.json >/dev/null

jq -r '.data.certificate' ${WORKDIR}/est/bootstrap.json > ${WORKDIR}/est/bootstrap.pem
jq -r '.data.private_key' ${WORKDIR}/est/bootstrap.json > ${WORKDIR}/est/bootstrap.key
openssl x509 -in ${WORKDIR}/est/bootstrap.pem -noout -subject -serial

echo
echo "=== sanity: cert auth login with bootstrap cert (does not replace VAULT_TOKEN) ==="
curl -k -sS --cert ${WORKDIR}/est/bootstrap.pem --key ${WORKDIR}/est/bootstrap.key \
  --request POST --data '{"name":"est-clients"}' \
  ${VAULT_ADDR}/v1/auth/est-cert/login \
  | jq '{policies:.auth.policies, token_type:.auth.token_type, cert_name:.auth.metadata.cert_name, common_name:.auth.metadata.common_name}'

CN="tls-device.est.example.com"
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/tls_device.key \
  -out ${WORKDIR}/est/tls_device.csr \
  -subj "/CN=${CN}" 2>/dev/null
openssl req -in ${WORKDIR}/est/tls_device.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/tls_device.p10

echo
echo "=== POST /.well-known/est/simpleenroll (TLS client cert, no Basic) ==="
curl -k -sS -D ${WORKDIR}/est/enroll_tls.hdr \
  -o ${WORKDIR}/est/tls_device.p7 \
  --cert ${WORKDIR}/est/bootstrap.pem \
  --key ${WORKDIR}/est/bootstrap.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/tls_device.p10 \
  "${EST_BASE}/simpleenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding|www-authenticate' ${WORKDIR}/est/enroll_tls.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/enroll_tls.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST TLS enroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/tls_device.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/tls_device.p7 ${WORKDIR}/est/tls_device.pem
openssl x509 -in ${WORKDIR}/est/tls_device.pem -out ${WORKDIR}/est/tls_device_leaf.pem

echo
echo "=== issued TLS-enrolled certificate ==="
openssl x509 -in ${WORKDIR}/est/tls_device_leaf.pem -noout -subject -issuer -serial -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/tls_device_leaf.pem
openssl x509 -in ${WORKDIR}/est/tls_device_leaf.pem -noout -serial | tee ${WORKDIR}/est/tls_device.serial

=== bootstrap leaf via PKI issue (admin token) ===
subject= /CN=bootstrap.est.example.com
serial=4B610F83A44006B0D42E7EBA5B1E7529E29D8C5A

=== sanity: cert auth login with bootstrap cert (does not replace VAULT_TOKEN) ===
{
  "policies": [
    "default",
    "est-enroll"
  ],
  "token_type": "batch",
  "cert_name": "est-clients",
  "common_name": "bootstrap.est.example.com"
}

=== POST /.well-known/est/simpleenroll (TLS client cert, no Basic) ===
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime; smime-type=certs-only

=== issued TLS-enrolled certificate ===
subject= /CN=tls-device.est.example.com
issuer= /CN=example.com EST Intermediate Authority
serial=013A2AFFE201ABC1F4ACE37F0F7085D63B3AD1B0
notBefore=Sep 18 13:45:22 2026 GMT
notAfter=Sep 19 01:45:52 2026 GMT
/tmp/vault/est/tls_device_leaf.pem: OK
serial=013A2AFFE201ABC1F4ACE37F0F7085D63B3AD1B0


### TLS — `simplereenroll` using the just-enrolled device cert

Same identity check as Basic reenroll ([RFC 7030 §4.2.2](https://www.rfc-editor.org/rfc/rfc7030#section-4.2.2)). Here the TLS client cert is also the Vault authenticator (`est-cert`): present the current leaf over TLS and send a rekey CSR whose Subject (`PrintableString`) and SAN match it.

In [49]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="tls-device.est.example.com"
OLD_SERIAL=$(cat ${WORKDIR}/est/tls_device.serial)

cat > ${WORKDIR}/est/reenroll_req.cnf <<EOF
[req]
distinguished_name = dn
req_extensions = ext
string_mask = nombstr
prompt = no
utf8 = no
[dn]
CN = ${CN}
[ext]
subjectAltName = DNS:${CN}
EOF
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/tls_device_re.key \
  -out ${WORKDIR}/est/tls_device_re.csr \
  -config ${WORKDIR}/est/reenroll_req.cnf \
  -reqexts ext
openssl req -in ${WORKDIR}/est/tls_device_re.csr -noout -text | grep -A1 "Subject Alternative Name"
openssl req -in ${WORKDIR}/est/tls_device_re.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/tls_device_re.p10

echo "=== POST /.well-known/est/simplereenroll (TLS client cert of current leaf) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_tls.hdr \
  -o ${WORKDIR}/est/tls_device_re.p7 \
  --cert ${WORKDIR}/est/tls_device_leaf.pem \
  --key ${WORKDIR}/est/tls_device.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/tls_device_re.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/reenroll_tls.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_tls.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST TLS reenroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/tls_device_re.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/tls_device_re.p7 ${WORKDIR}/est/tls_device_re.pem
openssl x509 -in ${WORKDIR}/est/tls_device_re.pem -out ${WORKDIR}/est/tls_device_re_leaf.pem

NEW_SERIAL=$(openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -serial)
echo "old ${OLD_SERIAL}"
echo "new ${NEW_SERIAL}"
openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -subject -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/tls_device_re_leaf.pem

if [ -z "${NEW_SERIAL}" ] || [ "${OLD_SERIAL}" = "${NEW_SERIAL}" ]; then
  echo "FAIL: TLS re-enroll did not issue a new serial"
  exit 1
fi

CERT_PUB=$(openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -pubkey | openssl md5)
KEY_PUB=$(openssl rsa -in ${WORKDIR}/est/tls_device_re.key -pubout 2>/dev/null | openssl md5)
if [ "${CERT_PUB}" != "${KEY_PUB}" ]; then
  echo "FAIL: renewed certificate does not match tls_device_re.key"
  exit 1
fi
echo "OK: TLS re-enroll issued a new serial bound to the new key"

cp ${WORKDIR}/est/tls_device_re_leaf.pem ${WORKDIR}/est/tls_device_leaf.pem
cp ${WORKDIR}/est/tls_device_re.key ${WORKDIR}/est/tls_device.key
echo "${NEW_SERIAL}" > ${WORKDIR}/est/tls_device.serial

Generating a 2048 bit RSA private key
......................................................................+++++
............................+++++
writing new private key to '/tmp/vault/est/tls_device_re.key'
-----


            X509v3 Subject Alternative Name: 
                DNS:tls-device.est.example.com
=== POST /.well-known/est/simplereenroll (TLS client cert of current leaf) ===
HTTP/2 200 
content-transfer-encoding: base64
content-type: application/pkcs7-mime; smime-type=certs-only
old serial=013A2AFFE201ABC1F4ACE37F0F7085D63B3AD1B0
new serial=40D0D6E2662859D3B33A6BC6E361290F8C589CAF
subject= /CN=tls-device.est.example.com
notBefore=Sep 18 13:45:22 2026 GMT
notAfter=Sep 19 01:45:52 2026 GMT
/tmp/vault/est/tls_device_re_leaf.pem: OK
OK: TLS re-enroll issued a new serial bound to the new key


## Step 8: Negative checks and issued inventory

Unauthenticated enroll must fail. Wrong Basic credentials must fail. Listing `pki_est/certs` should show the leaves created above.

In [50]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== enroll with no credentials (expect 401) ==="
curl -k -sS -o ${WORKDIR}/est/noauth.body -w "HTTP %{http_code}\n" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"
echo "body:"
head -c 400 ${WORKDIR}/est/noauth.body; echo

echo
echo "=== enroll with wrong password (expect 4xx) ==="
curl -k -sS -o ${WORKDIR}/est/badpass.body -w "HTTP %{http_code}\n" \
  --user "estuser:wrongpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"
echo "body:"
head -c 400 ${WORKDIR}/est/badpass.body; echo

echo
echo "=== labeled EST path /.well-known/est/est-clients/cacerts ==="
curl -k -sS -o /dev/null -w "HTTP %{http_code}\n" \
  "${EST_BASE}/est-clients/cacerts"

echo
echo "=== certificates stored on pki_est ==="
curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request LIST --silent \
  ${VAULT_ADDR}/v1/pki_est/certs | jq -r '.data.keys[]' | tee ${WORKDIR}/est/serials.txt

echo
echo "=== leaf CNs on pki_est ==="
while IFS= read -r SERIAL; do
  CN=$(curl -k --header "X-Vault-Token: $VAULT_TOKEN" --silent \
    ${VAULT_ADDR}/v1/pki_est/cert/${SERIAL} | jq -r '.data.certificate' \
    | openssl x509 -noout -subject -nameopt RFC2253 2>/dev/null)
  echo "${SERIAL}  ${CN}"
done < ${WORKDIR}/est/serials.txt

=== enroll with no credentials (expect 401) ===
HTTP 401
body:
permission denied: the client lacks sufficient authorization

=== enroll with wrong password (expect 4xx) ===
HTTP 401
body:
permission denied: the client lacks sufficient authorization

=== labeled EST path /.well-known/est/est-clients/cacerts ===
HTTP 200

=== certificates stored on pki_est ===
01:3a:2a:ff:e2:01:ab:c1:f4:ac:e3:7f:0f:70:85:d6:3b:3a:d1:b0
09:90:1b:f0:2f:c9:7b:90:63:39:8f:9e:0c:52:37:2a:ad:67:18:43
40:d0:d6:e2:66:28:59:d3:b3:3a:6b:c6:e3:61:29:0f:8c:58:9c:af
4b:61:0f:83:a4:40:06:b0:d4:2e:7e:ba:5b:1e:75:29:e2:9d:8c:5a
72:ea:52:e2:d4:dc:6a:c2:de:72:fc:2e:8a:3e:a1:9b:c1:b5:4f:5a

=== leaf CNs on pki_est ===
01:3a:2a:ff:e2:01:ab:c1:f4:ac:e3:7f:0f:70:85:d6:3b:3a:d1:b0  subject= CN=tls-device.est.example.com
09:90:1b:f0:2f:c9:7b:90:63:39:8f:9e:0c:52:37:2a:ad:67:18:43  subject= CN=device1.est.example.com
40:d0:d6:e2:66:28:59:d3:b3:3a:6b:c6:e3:61:29:0f:8c:58:9c:af  subject= CN=tls-device.est.example.com
4b:61:0f:83:a

## Clean UP EST